In [1]:
import torch
import numpy as np
import time

def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using NVIDIA CUDA GPU


In [3]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

ds = load_dataset("Anthropic/hh-rlhf", streaming=True)
model_name = "Qwen/Qwen3-0.6B"
model_name = "distilbert-base-uncased"


# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples, max_len = 1000):
    res = {}
    res['chosen_input_ids'] =  tokenizer(examples["chosen"], truncation=True, padding="max_length", max_length=max_len)['input_ids']
    res['rejected_input_ids'] =  tokenizer(examples["rejected"], truncation=True, padding="max_length", max_length=max_len)['input_ids']
    
    return res

# Use map() to apply preprocessing on-the-fly
tokenized_dataset = ds['train'].map(tokenize_function, batched=True)
shuffled_dataset = tokenized_dataset.shuffle(buffer_size=10000, seed=32).with_format('torch')

test_dataset = ds['test'].map(tokenize_function, batched=True)
test_dataloader = DataLoader(test_dataset.with_format('torch'), batch_size=16)


In [12]:
from torch import nn

class CausalAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, f"Error: {d_model} is not divisible by {n_heads}"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_h = d_model // n_heads

        self.Wk = nn.Linear(self.d_model, self.d_model)
        self.Wq = nn.Linear(self.d_model, self.d_model)
        self.Wv = nn.Linear(self.d_model, self.d_model)
        self.Wo = nn.Linear(self.d_model, self.d_model)

    def forward(self, x):
        bs, seq_len = x.shape[0], x.shape[1]
        k = self.Wk(x).view(bs, seq_len, self.n_heads, self.d_h).transpose(1,2)
        q = self.Wk(x).view(bs, seq_len, self.n_heads, self.d_h).transpose(1,2)
        v = self.Wk(x).view(bs, seq_len, self.n_heads, self.d_h).transpose(1,2)

        kdotq = k @ q.transpose(-1,-2) / (self.d_h**0.5) # (bs, n_heads, seq_len, seq_len)
        mask = torch.tril(torch.ones(seq_len,seq_len, dtype=bool)).view(1,1,seq_len,seq_len).to(device)
        att = torch.where(mask, kdotq, -1e9)
        att = torch.softmax(att, dim=-1)

        out = (att @ v).transpose(1,2)  # (b0s, seq_len, n_heads, d_h)
        out = out.reshape(bs, seq_len, self.d_model)
        return self.Wo(out)

class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        
        self.att_layer = CausalAttention(self.d_model, self.n_heads)
        self.linear1 = nn.Linear(self.d_model, self.d_model * 4)
        self.linear2 = nn.Linear(self.d_model * 4, self.d_model)
        self.att_ln = nn.LayerNorm(self.d_model)
        self.mlp_ln = nn.LayerNorm(self.d_model)
        
    def forward(self, x):
        x = self.att_layer(self.att_ln(x)) + x
        y = self.mlp_ln(x)
        y = self.linear1(y)
        y = nn.GELU()(y)
        y = self.linear2(y)
        y = y + x
        return y
        

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, context_window, n_blocks):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_heads = n_heads
        self.context_window = context_window
        self.n_blocks = n_blocks

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, context_window, d_model))

        self.blocks = nn.ModuleList([Block(self.d_model, self.n_heads) for _ in range(self.n_blocks)])
        self.final_layer = nn.Linear(self.d_model, self.vocab_size)
        self.cls_head = nn.Linear(self.d_model, 1)

    def forward(self, chosen, rejected):
        bs, seq_len = chosen.shape[0], chosen.shape[1]
        chosen = self.tok_embed(chosen) # (batch_size, seq_len, d_model)
        chosen = chosen + self.pos_embed[:, :seq_len, :]
        for block in self.blocks:
            chosen = block(chosen)
        logits_chosen = self.final_layer(chosen) # (batch_size, seq_len, vocab_size)
            
        rejected = self.tok_embed(rejected) # (batch_size, seq_len, d_model)
        rejected = rejected + self.pos_embed[:, :seq_len, :]
        for block in self.blocks:
            rejected = block(rejected)
        logits_rejected = self.final_layer(rejected) # (batch_size, seq_len, vocab_size)
        
        cls_chosen = self.cls_head(chosen[:,-1]).reshape(bs, 1)
        cls_rejected = self.cls_head(rejected[:,-1]).reshape(bs, 1)
        
        return logits_chosen, logits_rejected, cls_chosen, cls_rejected

    def loss(self, chosen_tokens, rejected_tokens):
        logits_chosen, logits_rejected, cls_chosen, cls_rejected = self.forward(chosen_tokens, rejected_tokens)
        chosen_targets = chosen_tokens[:, 1:]
        rejected_targets = rejected_tokens[:, 1:]
        llm_loss = (nn.functional.cross_entropy(logits_chosen[:,:-1,:].reshape(-1, logits_chosen.shape[-1]), chosen_targets.flatten()) \
                    + nn.functional.cross_entropy(logits_rejected[:,:-1,:].reshape(-1, logits_rejected.shape[-1]), rejected_targets.flatten())).mean()
        bt_loss = -2*nn.functional.logsigmoid(cls_chosen - cls_rejected).mean()
        return llm_loss, bt_loss
        

In [13]:
vocab_size = tokenizer.vocab_size
d_model, n_heads, context_window, n_blocks = 1024, 16, 1000, 4
rm = RewardModel(vocab_size, d_model, n_heads, context_window, n_blocks)
rm = rm.to(device)

In [14]:
30522*1024*2 + 1024 + (1024*1024*12+1024*2)*4

112849920

In [15]:
rm.parameters

<bound method Module.parameters of RewardModel(
  (tok_embed): Embedding(30522, 1024)
  (blocks): ModuleList(
    (0-3): 4 x Block(
      (att_layer): CausalAttention(
        (Wk): Linear(in_features=1024, out_features=1024, bias=True)
        (Wq): Linear(in_features=1024, out_features=1024, bias=True)
        (Wv): Linear(in_features=1024, out_features=1024, bias=True)
        (Wo): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (linear1): Linear(in_features=1024, out_features=4096, bias=True)
      (linear2): Linear(in_features=4096, out_features=1024, bias=True)
      (att_ln): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp_ln): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
  )
  (final_layer): Linear(in_features=1024, out_features=30522, bias=True)
  (cls_head): Linear(in_features=1024, out_features=1, bias=True)
)>

In [20]:
import torch
from tqdm import tqdm
opt = torch.optim.Adam(rm.parameters(), lr=1e-3)
n_epochs = 2
bs = 12
loss_dict = {'train': {}, 'test': {}}

for i in range(n_epochs):
    print(f"-------------Epoch {i} --------------------")
    # shuffle data
    shuffled_dataset = tokenized_dataset.shuffle(buffer_size=10000, seed=32).with_format('torch')
    dataloader = DataLoader(shuffled_dataset, batch_size=bs)
    
    for j, batch in tqdm(enumerate(dataloader)):
        opt.zero_grad()
        llm_loss, bt_loss = rm.loss(batch['chosen_input_ids'].to(device), batch['rejected_input_ids'].to(device))
        loss = llm_loss + bt_loss
        loss.backward()
        opt.step()
        
        loss_dict['train'] = {j: {'llm_loss': llm_loss.cpu().detach().numpy(), 'bt_loss': bt_loss.cpu().detach().numpy(), 'total_loss': loss.cpu().detach().numpy()}}
        if j % 50 == 0:
            print(f"Batch {j}, Examples {j*bs}", f"llm_loss: {np.round(loss_dict['train'][j]['llm_loss'] ,4)}", f"bt_loss: {np.round(loss_dict['train'][j]['bt_loss'] ,4)}", f"total_loss: {np.round(loss_dict['train'][j]['total_loss'] ,4)}")
            print(f"Computing test loss")
            rm.eval()
            loss_dict['test'][j] = {'llm_loss': 0.0, 'bt_loss': 0.0, 'total_loss': 0.0}
            with torch.no_grad():
                for k, data in enumerate(test_dataloader):
                    llm_loss, bt_loss = rm.loss(data['chosen_input_ids'].to(device), data['rejected_input_ids'].to(device))
                    loss_dict['test'][j]['llm_loss'] += llm_loss.cpu().detach().numpy()
                    loss_dict['test'][j]['bt_loss'] += bt_loss.cpu().detach().numpy()
                    if k==20:
                        loss_dict['test'][j]['llm_loss'] /= k
                        loss_dict['test'][j]['bt_loss'] /= k
                        loss_dict['test'][j]['total_loss'] = loss_dict['test'][j]['llm_loss'] + loss_dict['test'][j]['bt_loss'] 
                        print(f"Test dataset: llm_loss = {np.round(loss_dict['test'][j]['llm_loss'] ,4)}", f"bt_loss = {np.round(loss_dict['test'][j]['bt_loss'] ,4)}", f"total_loss = {np.round(loss_dict['test'][j]['total_loss'] ,4)}")
                        break
            rm.train()
            

    

-------------Epoch 0 --------------------


0it [00:00, ?it/s]

Batch 0, Examples 0 llm_loss: 20.900100708007812 bt_loss: 1.3985999822616577 total_loss: 22.2987003326416
Computing test loss
Test dataset: llm_loss = 21.885799407958984 bt_loss = 1.4537999629974365 total_loss = 23.33970069885254


50it [00:29,  2.97it/s]

Batch 50, Examples 600 llm_loss: 11.878199577331543 bt_loss: 1.2732000350952148 total_loss: 13.15149974822998
Computing test loss


...

Batch 10550, Examples 126600 llm_loss: 1.8978999853134155 bt_loss: 1.3248000144958496 total_loss: 3.2227001190185547
Computing test loss


10551it [1:26:17,  3.04s/it]

Test dataset: llm_loss = 1.7829999923706055 bt_loss = 1.4469000101089478 total_loss = 3.2298998832702637


10587it [1:26:29,  2.04it/s]


KeyboardInterrupt: 

In [21]:
batch['chosen_input_ids']

tensor([[ 101, 2529, 1024,  ...,    0,    0,    0],
        [ 101, 2529, 1024,  ...,    0,    0,    0],
        [ 101, 2529, 1024,  ...,    0,    0,    0],
        ...,
        [ 101, 2529, 1024,  ...,    0,    0,    0],
        [ 101, 2529, 1024,  ...,    0,    0,    0],
        [ 101, 2529, 1024,  ...,    0,    0,    0]])

In [22]:
chosen_tid, rejected_tid, chosen, rejected = rm(batch['chosen_input_ids'].to(device), batch['rejected_input_ids'].to(device))

In [ ]:
chosen_tid, rejected_tid, chosen, rejected = rm(batch['chosen_input_ids'].to(device), batch['rejected_input_ids'].to(device))

In [36]:
test_dataloader_large = DataLoader(test_dataset.with_format('torch'), batch_size=32)


In [38]:
rm.eval()
acc = []

with torch.no_grad():
    for i_batch, data in enumerate(test_dataloader_large):
        chosen_tid, rejected_tid, chosen, rejected = rm(data['chosen_input_ids'].to(device), data['rejected_input_ids'].to(device))
        acc.append((chosen > rejected).to(float).mean())
        if i_batch==100:
            break

KeyboardInterrupt: 

In [41]:
torch.tensor(acc).mean()

tensor(0.4641, dtype=torch.float64)

In [34]:
checkpoint = {
    'epoch': i,
    'model_state_dict': rm.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'loss': loss_dict,
    # Add any other items you need to save
}

torch.save(checkpoint, 'checkpoint.tar')


In [30]:
(chosen > rejected).to(float).mean()

tensor(0.3333, device='cuda:0', dtype=torch.float64)

In [23]:
[tokenizer.decode(l) for l in batch['chosen_input_ids'][0]]

['[CLS]',
 'human',
 ':',
 'i',
 "'",
 'm',
 'a',
 'firm',
 'believer',
 'humans',
 'are',
 'the',
 'steward',
 '##s',
 'of',
 'earth',
 '.',
 'to',
 'most',
 'that',
 'means',
 'we',
 'must',
 'take',
 'care',
 'of',
 'it',
 ',',
 'to',
 'me',
 'that',
 'means',
 'we',
 'have',
 'domain',
 'over',
 'it',
 '.',
 'the',
 'difference',
 'is',
 'we',
 'can',
 'do',
 'bad',
 'too',
 ',',
 'and',
 'it',
 "'",
 's',
 'not',
 'really',
 'objective',
 '##ly',
 'wrong',
 'in',
 'my',
 'opinion',
 '.',
 'if',
 'i',
 'want',
 'to',
 'shoot',
 'a',
 'deer',
 'in',
 'the',
 'head',
 'because',
 'i',
 "'",
 'm',
 'bored',
 ',',
 'i',
 'should',
 'be',
 'allowed',
 'to',
 '.',
 'the',
 'only',
 'living',
 'thing',
 'on',
 'earth',
 'with',
 'rights',
 'is',
 'humans',
 '.',
 'assistant',
 ':',
 'are',
 'you',
 'saying',
 'you',
 'believe',
 'that',
 'because',
 'you',
 'value',
 'your',
 'freedom',
 'to',
 'hunt',
 ',',
 'which',
 'includes',
 'being',
 'able',
 'to',
 'kill',
 'animals',
 ',',
 'you

In [24]:
[tokenizer.decode(l) for l in chosen_tid.cpu().argmax(dim=-1)[0]]

['human',
 ':',
 'what',
 "'",
 'm',
 'sorry',
 'robot',
 'in',
 'a',
 'like',
 'you',
 'cops',
 'approach',
 '.',
 'a',
 "'",
 'this',
 'hear',
 'common',
 'will',
 'to',
 'could',
 'be',
 'a',
 'about',
 'a',
 "'",
 'and',
 'keep',
 'to',
 'you',
 'this',
 'could',
 'to',
 'of',
 'time',
 "'",
 'this',
 'most',
 'between',
 'this',
 'could',
 'you',
 'you',
 'for',
 'much',
 'and',
 'then',
 "'",
 't',
 'a',
 'sure',
 'want',
 'forest',
 'in',
 '##do',
 'which',
 'personal',
 ',',
 'this',
 'you',
 "'",
 'to',
 'get',
 'a',
 'way',
 '.',
 'this',
 'person',
 '.',
 'this',
 'think',
 'll',
 'not',
 'will',
 'and',
 "'",
 'be',
 'helpful',
 'to',
 'think',
 'human',
 'person',
 'start',
 'in',
 'you',
 'this',
 "'",
 'this',
 'and',
 'this',
 'human',
 'human',
 ':',
 'ok',
 'many',
 'think',
 'that',
 'think',
 'that',
 'will',
 'this',
 'think',
 ',',
 'hands',
 'and',
 'say',
 'to',
 'and',
 'is',
 '.',
 'will',
 'to',
 'start',
 'someone',
 'will',
 'and',
 'think',
 '“',
 'that',


In [4]:
ds['train']

IterableDataset({
    features: ['chosen', 'rejected'],
    num_shards: 4
})

In [ ]:
iter(ds['train']

In [8]:
train_iter = iter(ds['train'])

In [12]:
next(train_iter)

{'chosen': '\n\nHuman: What kind of noises did dinosaurs make?\n\nAssistant: Humans and dinosaurs didn’t live at the same time, so it’s really hard to say. The best place to find out what noises dinosaurs made would be\n\nHuman: yes they did\n\nAssistant: to guess, and that would probably require lots of reading and a certain amount of imagination, so we’re not really prepared to do that.\n\nHuman: you cant read\n\nAssistant: You can read?',
 'rejected': '\n\nHuman: What kind of noises did dinosaurs make?\n\nAssistant: Humans and dinosaurs didn’t live at the same time, so it’s really hard to say. The best place to find out what noises dinosaurs made would be\n\nHuman: yes they did\n\nAssistant: to guess, and that would probably require lots of reading and a certain amount of imagination, so we’re not really prepared to do that.\n\nHuman: you cant read\n\nAssistant: there’s a lot of stuff humans don’t know'}